# **Subquestion 2 Dataset: Federal  Reserve Survey**

Dataset from https://www.federalreserve.gov/econres/scfindex.htm

Dataset Summary (Survey of Consumer Finances – 2022)

This analysis uses data from the Survey of Consumer Finances (SCF), conducted by the Federal Reserve. The SCF is a nationally representative survey of U.S. households that provides detailed information on income, assets, debts, and financial behavior.

Key features:

* Includes household-level data on wealth components (housing, financial assets, businesses, debt)
* Uses multiple imputations (5 implicates) to handle missing or sensitive responses
* Oversamples high-wealth households to better capture the upper tail of the wealth distribution
* Widely used for research on inequality, wealth composition, and financial decision-making

The dataset enables detailed analysis of how different asset types relate to household income and financial outcomes.

Libraries Used

In [ ]:
from operator import index
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_curve, auc
from sklearn.ensemble import RandomForestClassifier

Process and Merge *Main Survey Data* and *Replicate Weight File* raw data

In [ ]:
main_survey_df = pd.read_stata("p22i6.dta")
rwf_df = pd.read_stata("p22_rw1.dta")

rwf_df = rwf_df.drop_duplicates(subset=['y1'])

df = main_survey_df.merge(rwf_df, on="y1", how="left")

df = df.sort_values('y1').reset_index(drop=True)

df['yy1'] = df.index // 5
df['imp'] = df.index % 5 + 1

Load Codebook

In [ ]:
varmap = pd.read_excel("2022map.xlsx")
varmap.columns = ['var', 'description', 'c3', 'c4', 'index']
varmap.dropna(subset=['description'], inplace=True)

Clean and rename variables. Select relevant variables to use.

In [ ]:
# Clean strings
varmap['description'] = (
    varmap['description']
    .str.split(': ', n=1)
    .str[-1]
    .str.strip()
)

# Clean df columns
varmap['var'] = varmap['var'].str.lower().str.strip()

df.columns = df.columns.str.lower().str.strip()

# Mapping + rename
mapping = dict(zip(varmap['var'], varmap['description']))
df = df.rename(columns=mapping)

df_model = df[[
    'imp',
    'PUBLIC DATASET CASE ID NUMER',
    #'X30054', # X30054: ZIP CODE
    #'X32110', # X32110: MEDIAN INCOME OF CENSUS TRACT
    #'X32130', # X32130: MEDIAN HOUSE VALUE IN CENSUS TRACT
    #'X32135', # X32135: MEDIAN RENTAL VALUE IN CENSUS TRACT
    # Demographics / controls
    'R: RECONCILED AGE',
    #'# OF PEOPLE IN PEU',

    # Income'
    'TOTAL INCOME',
    'PAST 5 YRS, INC HGHR, LWR, SAME?',
    'RE_1: NET INCOM REC',
    # Homeownership / housing
    'HU_OTH: OWN/RENT CONDO/CO-OP/OTH HU?',
    'HU_OTH: OWN ENTIRE BUILDING OR UNIT?',
    'HOW IS HOUSING PROVIDED?', # if not a homeowner, how is housing provided
    'HU_OTH: CURR VAL HM/LAND', # primary residence dollar amount
    'MOBL_OWNHM_&_SIT: ORIG COST BOTH', # mobile home dollar amount
    # Mortgage / housing debt
    'MORT_1: HAVE MORT/LAND CONTRACT?',
    'MORT_1: TOTAL AMT BORROWED/REFINANCED',
    'MORT_1: AMT STILL OWED',

    # Financial assets
    'HAVE CHKING ACCTS?',
    'CHKING_1: AMT ACCT',

    'MOPUP: CHKING: AMT REMAIN ACCTS',
    "ASSET IN R'S HOME",
    'ASSET IN STOCKS',
    'TOT MARKET VAL STOCKS', # HERE
    'ASSET IN BONDS',
    'TOT MKT VAL ALL BONDS', #HERE
    'ASSET IN MUTUAL FUNDS',
    'ASSET IN MONEY MARKET',
    'ASSET IN BUSINESS',
    'ASSET IN OTHER REAL ESTATE',

    # Other assets
    'RE_1: WORTH IF SOLD TODAY', # other properties
    'MOPUP: OWN_VEH: TOT VAL REMAIN VEHS',
    'MOPUP: BUS: VAL REMAINING BUS',

    # Debt & liabilities
    'CC_BANK: AMT STILL OWE',
    'MOPUP: OWN_VEH: TOT AMT OWED ON REMAIN VEH',
    'COMPUTED VALUE - # OF LINES OF CREDIT',
    'LOC_1: SECURED BY HM EQUITY?',
    'LOC_1: MAXIMUM AMT CAN BORROW',
    'LOC_1: AMT OWED AGAINST LINE',

    # Behavior
    'FIN RISK WILLINGNESS'
]]

rename_dict = {
    'imp': 'implicate',

    'PUBLIC DATASET CASE ID NUMER': 'case_id',
    'R: RECONCILED AGE': 'age',
    'TOTAL INCOME': 'total_income',
    'PAST 5 YRS, INC HGHR, LWR, SAME?': 'income_change_5yr',

    'RE_1: NET INCOM REC': 'real_estate_income_1',

    'HU_OTH: OWN/RENT CONDO/CO-OP/OTH HU?': 'housing_type_other',
    'HU_OTH: OWN ENTIRE BUILDING OR UNIT?': 'own_building_or_unit',
    'HOW IS HOUSING PROVIDED?': 'housing_tenure',
    'HU_OTH: CURR VAL HM/LAND': 'home_value',
    'MOBL_OWNHM_&_SIT: ORIG COST BOTH': 'mobile_home_value',

    'MORT_1: HAVE MORT/LAND CONTRACT?': 'has_mortgage',
    'MORT_1: TOTAL AMT BORROWED/REFINANCED': 'mortgage_total_borrowed',
    'MORT_1: AMT STILL OWED': 'mortgage_balance',

    'HAVE CHKING ACCTS?': 'has_checking',
    'CHKING_1: AMT ACCT': 'checking_balance_1',
    'MOPUP: CHKING: AMT REMAIN ACCTS': 'checking_balance_other',

    "ASSET IN R'S HOME": 'home_equity',
    'ASSET IN STOCKS': 'stocks_value',
    'ASSET IN BONDS': 'bonds_value',
    'ASSET IN MUTUAL FUNDS': 'mutual_funds_value',
    'ASSET IN MONEY MARKET': 'money_market_value',
    'ASSET IN BUSINESS': 'business_value',
    'ASSET IN OTHER REAL ESTATE': 'other_real_estate_value',

    'RE_1: WORTH IF SOLD TODAY': 'real_estate_value_1',
    'MOPUP: OWN_VEH: TOT VAL REMAIN VEHS': 'vehicle_value_total',
    'MOPUP: BUS: VAL REMAINING BUS': 'business_value_other',

    'CC_BANK: AMT STILL OWE': 'credit_card_balance',
    'MOPUP: OWN_VEH: TOT AMT OWED ON REMAIN VEH': 'vehicle_debt_total',

    'COMPUTED VALUE - # OF LINES OF CREDIT': 'num_credit_lines',
    'LOC_1: SECURED BY HM EQUITY?': 'heloc_secured',
    'LOC_1: MAXIMUM AMT CAN BORROW': 'heloc_limit',
    'LOC_1: AMT OWED AGAINST LINE': 'heloc_balance',

    'FIN RISK WILLINGNESS': 'risk_tolerance'
}


Data preview

In [ ]:
print(df.head())

In [ ]:
df.describe()

Rename Variables + Create Core Financial Measures

In [ ]:
df_model = df_model.rename(columns=rename_dict)
df_model['checking_total'] = (
    df_model['checking_balance_1'].fillna(0) +
    df_model['checking_balance_other'].fillna(0)
)

df_model['stocks'] = df_model['TOT MARKET VAL STOCKS'].fillna(0)
df_model['bonds'] = df_model['TOT MKT VAL ALL BONDS'].fillna(0)

df_model['financial_assets'] = (
    df_model['checking_total'] +
    df_model['stocks'] +
    df_model['bonds']
)

print(df_model[['stocks','bonds','financial_assets']].describe())

Data Cleaning + Feature Engineering (ML Dataset)

In [ ]:
df_ml = df_model.copy()
df_ml = df_ml.replace([-1, -7, -8, -9], np.nan)

df_ml['home_equity_true'] = (
    df_ml['primary_home_value'] -
    pd.to_numeric(df_ml['mortgage_balance'], errors='coerce').fillna(0)
).clip(lower=0)

df_ml['checking_total'] = (
    pd.to_numeric(df_ml['checking_balance_1'], errors='coerce').fillna(0) +
    pd.to_numeric(df_ml['checking_balance_other'], errors='coerce').fillna(0)
)

df_ml['stocks'] = pd.to_numeric(df_ml['TOT MARKET VAL STOCKS'], errors='coerce').fillna(0)
df_ml['bonds'] = pd.to_numeric(df_ml['TOT MKT VAL ALL BONDS'], errors='coerce').fillna(0)

df_ml['financial_assets'] = df_ml['checking_total'] + df_ml['stocks'] + df_ml['bonds']

df_ml['financial_assets_full'] = (
    df_ml['checking_raw'] +
    df_ml['stocks_raw'] +
    df_ml['bonds_raw'] +
    df_ml['mutual_funds'] +
    df_ml['money_market'] +
    df_ml['business']
)

df_ml['total_assets'] = (
    df_ml['home_equity_true'] +
    df_ml['financial_assets_full']
)

df_ml['primary_home_value'] = (
    pd.to_numeric(df_ml['home_value'], errors='coerce').fillna(0) +
    pd.to_numeric(df_ml['mobile_home_value'], errors='coerce').fillna(0)
)



df_ml['share_home_equity'] = df_ml['home_equity_true'] / df_ml['total_assets']
df_ml['share_checking'] = df_ml['checking_raw'] / df_ml['total_assets']
df_ml['share_stocks'] = df_ml['stocks_raw'] / df_ml['total_assets']
df_ml['share_bonds'] = df_ml['bonds_raw'] / df_ml['total_assets']
df_ml['share_mutual_funds'] = df_ml['mutual_funds'] / df_ml['total_assets']
df_ml['share_money_market'] = df_ml['money_market'] / df_ml['total_assets']
df_ml['share_business'] = df_ml['business'] / df_ml['total_assets']

df_ml = df_ml.dropna(subset=['home_equity_true','financial_assets','total_income'])

df_ml['income_group'] = pd.qcut(df_ml['total_income'], 4, labels=['Low','Lower-Mid','Upper-Mid','High'])

Winsorize key variables at the 99th percentile to reduce the influence of extreme outliers

In [ ]:
for col in ['home_equity_true','financial_assets','total_income']:
    df_ml[col] = df_ml[col].clip(lower=0)
    df_ml[col] = df_ml[col].clip(upper=df_ml[col].quantile(0.99))

### **EDA**

Select columns for correlation analysis

In [ ]:
cols = [
    'share_home_equity',
    'share_checking',
    'share_stocks',
    'share_bonds',
    'share_mutual_funds',
    'share_money_market',
    'share_business',
    'total_income'
]

corr_df = df_ml[cols].copy()

Correlation heatmap

In [ ]:
corr = corr_df.corr(method='pearson')

fig, ax = plt.subplots(figsize=(8, 6))

im = ax.imshow(corr.values, cmap='Reds', vmin=-1, vmax=1)

ax.set_xticks(np.arange(len(corr.columns)))
ax.set_yticks(np.arange(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=45, ha='right')
ax.set_yticklabels(corr.columns)

for i in range(len(corr)):
    for j in range(len(corr)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}",
                ha='center', va='center', fontsize=9)

ax.set_title("Correlation of Asset Composition and Income")
cbar = plt.colorbar(im, ax=ax)
cbar.set_label('Correlation', rotation=270, labelpad=15)

plt.tight_layout()
plt.show()

**Distribution of asset between homeowners vs non-homeowners (renters)**

Total financial assets and total household assets, filters valid observations, and calculate asset share ratios for each component.

In [ ]:
df_ml['is_homeowner'] = (df_ml['primary_home_value'] > 0).astype(int)

df_ml['checking_raw'] = pd.to_numeric(df_ml['checking_total'], errors='coerce').fillna(0)
df_ml['stocks_raw'] = pd.to_numeric(df_ml['stocks'], errors='coerce').fillna(0)
df_ml['bonds_raw'] = pd.to_numeric(df_ml['bonds'], errors='coerce').fillna(0)
df_ml['mutual_funds'] = pd.to_numeric(df_ml['mutual_funds_value'], errors='coerce').fillna(0)
df_ml['money_market'] = pd.to_numeric(df_ml['money_market_value'], errors='coerce').fillna(0)
df_ml['business'] = pd.to_numeric(df_ml['business_value'], errors='coerce').fillna(0)

df_ml['financial_assets_full'] = (
    df_ml['checking_raw'] +
    df_ml['stocks_raw'] +
    df_ml['bonds_raw'] +
    df_ml['mutual_funds'] +
    df_ml['money_market'] +
    df_ml['business']
)

df_ml['total_assets'] = (
    df_ml['home_equity_true'] +
    df_ml['financial_assets_full']
)

df_ml = df_ml[df_ml['total_assets'] > 0]

df_ml['share_home_equity'] = df_ml['home_equity_true'] / df_ml['total_assets']
df_ml['share_checking'] = df_ml['checking_raw'] / df_ml['total_assets']
df_ml['share_stocks'] = df_ml['stocks_raw'] / df_ml['total_assets']
df_ml['share_bonds'] = df_ml['bonds_raw'] / df_ml['total_assets']
df_ml['share_mutual_funds'] = df_ml['mutual_funds'] / df_ml['total_assets']
df_ml['share_money_market'] = df_ml['money_market'] / df_ml['total_assets']
df_ml['share_business'] = df_ml['business'] / df_ml['total_assets']

Calculates median asset shares by homeownership status

In [ ]:
summary = df_ml.groupby('is_homeowner')[[
    'share_home_equity',
    'share_checking',
    'share_stocks',
    'share_bonds',
    'share_mutual_funds',
    'share_money_market',
    'share_business'
]].median()

summary.index = ['Renter', 'Homeowner']
summary = summary.round(3)

Computes mean asset shares by homeownership status and outputs both conditional stock shares

In [ ]:
means = df_ml.groupby('is_homeowner')[[
    'share_home_equity',
    'share_checking',
    'share_stocks',
    'share_bonds',
    'share_mutual_funds',
    'share_money_market',
    'share_business'
]].mean()

print(stocks_conditional)
print(means)

### **Logistic Model + ROC Analysis by Income Group**

Train Logistic Regression model

In [ ]:
def run_model(X, y):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    model = LogisticRegression(max_iter=1000)
    model.fit(X_train, y_train)
    prob = model.predict_proba(X_test)[:,1]
    fpr, tpr, _ = roc_curve(y_test, prob)
    return fpr, tpr, auc(fpr, tpr)

Divide income groups into 4

In [ ]:
df_ml['income_group'] = pd.qcut(
    df_ml['total_income'],
    4,
    labels=['Low','Lower-Mid','Upper-Mid','High']
)

# Compute ranges
income_ranges = df_ml.groupby('income_group', observed=True)['total_income'].agg(['min','max'])

def format_range(row):
    return f"${row['min']:,.0f} - ${row['max']:,.0f}"

income_ranges['label'] = income_ranges.apply(format_range, axis=1)

groups = ['Low', 'Lower-Mid', 'Upper-Mid', 'High']


Evaluate how well different asset types distinguish higher- vs lower-income households within each income bracket. Models are estimated separately by income quartile to control for structural differences across the income distribution.

* Model performance is measured using AUC. Higher AUC ⇒ stronger within-group predictor of income.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11,8))
axes = axes.flatten()

for i, g in enumerate(groups):
    subset = df_ml[df_ml['income_group'] == g]

    cutoff = subset['total_income'].median()
    y = (subset['total_income'] > cutoff).astype(int)

    fpr_h, tpr_h, auc_h = run_model(subset[['home_equity_true']], y)
    fpr_f, tpr_f, auc_f = run_model(subset[['financial_assets']], y)

    axes[i].plot(fpr_h, tpr_h, linewidth=2.5, label=f'Home Equity (AUC={auc_h:.2f})')
    axes[i].plot(fpr_f, tpr_f, linewidth=2.5, label=f'Financial Assets (AUC={auc_f:.2f})')
    axes[i].plot([0,1],[0,1],'k--', alpha=0.6)

    # ADD RANGE TO TITLE
    axes[i].set_title(
        f"{g} Income\n({income_ranges.loc[g, 'label']})",
        fontsize=11
    )

    axes[i].set_xlabel('False Positive Rate')
    axes[i].set_ylabel('True Positive Rate')
    axes[i].legend(frameon=False, fontsize=8)
    axes[i].grid(alpha=0.2)

plt.suptitle('Asset Predictive Power Across Income Levels', fontsize=14)
plt.tight_layout()
plt.show()

### **Random Forest: Feature Importance**

Split, train, and used Random Forest to identify which household assets best predict high-income status. Inputs include home equity, liquid assets, and investment exposure.

Adjustments:
1.  Outliers capped at the 99th percentile
2.  Log transforms applied to highly skewed asset values
3.  Ownership indicators included to capture extensive margins

In [ ]:
threshold = df_ml['total_income'].quantile(0.7)
df_ml['high_income'] = (df_ml['total_income'] > threshold).astype(int)

features = ['home_equity_true','financial_assets']
X = df_ml[features]
y = df_ml['high_income']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


model = RandomForestClassifier(n_estimators=200, random_state=42)
model.fit(X_train, y_train)

for col in ['home_equity_true','checking_total','stocks','bonds','total_income']:
    df_ml[col] = df_ml[col].clip(lower=0)
    df_ml[col] = df_ml[col].clip(upper=df_ml[col].quantile(0.99))

df_ml['stocks_log'] = np.log1p(df_ml['stocks'])
df_ml['bonds_log'] = np.log1p(df_ml['bonds'])

df_ml['has_stocks'] = (df_ml['stocks'] > 0).astype(int)
df_ml['has_bonds'] = (df_ml['bonds'] > 0).astype(int)

df_ml = df_ml.dropna(subset=[
    'home_equity_true',
    'checking_total',
    'stocks_log',
    'bonds_log',
    'total_income'
])

threshold = df_ml['total_income'].quantile(0.7)
df_ml['high_income'] = (df_ml['total_income'] > threshold).astype(int)

features = [
    'home_equity_true',
    'checking_total',
    'stocks_log',
    'has_stocks',
    'bonds_log'
]

X = df_ml[features]
y = df_ml['high_income']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = RandomForestClassifier(n_estimators=300, random_state=42)
model.fit(X_train, y_train)

importances = model.feature_importances_

Plot feature importance as bar graph

In [ ]:
feat_df = pd.DataFrame({
    'feature': X.columns,
    'importance': importances
}).sort_values(by='importance')

label_map = {
    'home_equity_true': 'Home Equity ($)',
    'checking_total': 'Checking ($)',
    'stocks_log': 'Stocks (log $)',
    'has_stocks': 'Owns Stocks',
    'bonds_log': 'Bonds (log $)'
}

feat_df['feature'] = feat_df['feature'].map(label_map)

plt.figure(figsize=(8,5))

colors = ['#4E79A7','#F28E2B','#59A14F','#E15759','#76B7B2']

bars = plt.barh(
    feat_df['feature'],
    feat_df['importance'],
    color=colors[:len(feat_df)]
)

for bar in bars:
    width = bar.get_width()
    plt.text(width + 0.01, bar.get_y() + bar.get_height()/2,
             f"{width:.2f}", va='center', fontsize=10)

plt.xlabel('Relative Importance', fontsize=11)
plt.title('Which Assets Best Predict Household Income', fontsize=13, weight='bold')

plt.xlim(0, max(feat_df['importance']) * 1.25)

plt.grid(axis='x', linestyle='--', alpha=0.3)

plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)
plt.gca().spines['left'].set_visible(False)

plt.tight_layout()
plt.show()
